# Analisis Exploratorio de la Encuesta Nacional para el Sistema de Cuidados (ENASIC) 2022

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
import pathlib
from scipy.stats import norm

## carga del dataset

In [2]:


# 1. Definir la ruta relativa de tu archivo ZIP
ruta_relativa_zip = "../../data/bronze/ENASIC/enasic_2022_bd_csv.zip"


diccionario_dfs = {}


with zipfile.ZipFile(ruta_relativa_zip, 'r') as archivo_zip:
    
    
    for nombre_archivo in archivo_zip.namelist():
        
        
        if nombre_archivo.endswith('.csv'):
            
            
            with archivo_zip.open(nombre_archivo) as archivo_csv:
                

                df = pd.read_csv(archivo_csv)
                

                diccionario_dfs[nombre_archivo.split(".")[0].lower()] = df
                print(f" Cargado: {nombre_archivo} con {len(df)} filas.")



 Cargado: TVIVIENDA.csv con 6423 filas.
 Cargado: TCSDEMPO.csv con 21776 filas.
 Cargado: THOG_UNIP.csv con 928 filas.
 Cargado: THOGAR.csv con 6508 filas.
 Cargado: TPER_ELE.csv con 5579 filas.
 Cargado: TPOB_CUI.csv con 5677 filas.


In [3]:
diccionario_dfs.keys()

dict_keys(['tvivienda', 'tcsdempo', 'thog_unip', 'thogar', 'tper_ele', 'tpob_cui'])

In [4]:
for name,df in diccionario_dfs.items():
    print(f"{name}: {df.shape} \n")
    

tvivienda: (6423, 44) 

tcsdempo: (21776, 181) 

thog_unip: (928, 315) 

thogar: (6508, 72) 

tper_ele: (5579, 275) 

tpob_cui: (5677, 432) 



In [ ]:
# vemos una gran cantidad de columnas el archivo con menor columnas es de 44, ver referencia de INEGI en xlsx

In [ ]:
# en esta parte se corre codigo R que viene como un anexo en un archivo pdf referente a la encuesta
# 


## Por Validar

In [5]:
tsd = diccionario_dfs["tcsdempo"]

In [7]:
"""
================================================================
Construcción de la población susceptible que recibe cuidados
ENASIC 2022 — Traducción de R a Python
================================================================
Fuente original: INEGI / ENASIC 2022
Tabla principal: TCSDEMPO.csv
================================================================
"""

# import pandas as pd
# import numpy as np
from functools import reduce

# ---------------------------------------------------------------
# 1. Carga de datos
# ---------------------------------------------------------------
# Ajusta la ruta según tu directorio de trabajo
# tsd = pd.read_csv("TCSDEMPO.csv")

# Convertir EDAD a numérico (equivalente a as.numeric(as.character()))
tsd["EDAD"] = pd.to_numeric(tsd["EDAD"], errors="coerce")

# ---------------------------------------------------------------
# 2. Función auxiliar para extraer subcadenas
#    Equivalente a substr_M() en R
# ---------------------------------------------------------------
def substr_m(x, n, lado):
    """
    Extrae n caracteres de una cadena.
    lado='D' → desde la derecha (últimos n caracteres)
    lado='I' → desde la izquierda (primeros n caracteres)
    """
    x = str(x)
    if lado == "D":
        return x[-n:]
    if lado == "I":
        return x[:n]

# ---------------------------------------------------------------
# 3. Construcción del número de renglón e ID_AGE
#    N_REN: últimos 2 dígitos de LLAVESDE
#    ID_AGE: LLAVEHOG + "." + N_REN (enlaza cuidador con su edad)
# ---------------------------------------------------------------
tsd["N_REN"] = (
    tsd["LLAVESDE"]
    .astype(str)
    .str[-2:]          # equivalente a substr_M(LLAVESDE, 2, "D")
    .astype(int)
)

tsd["ID_AGE"] = (
    tsd["LLAVEHOG"].astype(str)
    + "."
    + tsd["N_REN"].astype(str)
)

# ---------------------------------------------------------------
# 4. Función auxiliar reutilizable
#    Para cada subpoblación, el proceso es idéntico:
#    - Filtra filas donde la variable de cuidador no es NA/0/99
#    - Construye ID_AGE del cuidador reportado
#    - Hace merge para obtener la edad del cuidador
#    - Guarda solo LLAVESDE y la columna de edad resultante
# ---------------------------------------------------------------
# Tabla auxiliar de búsqueda de edades (LLAVESDE → ID_AGE → EDAD)
edad_lookup = tsd[["ID_AGE", "EDAD"]].copy()

# Rango válido de edad de cuidadores (15 a 98 años inclusive)
VALID_AGES = list(range(15, 99))

def get_caregiver_ages(tsd, var_list):
    """
    Para cada variable en var_list (número de renglón del cuidador):
    1. Filtra registros válidos (no NA, no 0, no 99)
    2. Construye el ID_AGE del cuidador
    3. Hace merge con la tabla de edades
    4. Devuelve lista de DataFrames con [LLAVESDE, <var>_ED]

    Equivalente al bucle for con merge en R.
    """
    frames = []
    for var in var_list:
        # Filtrar: excluir NA, 0 y 99 (equivalente a !%in%c(NA,0,99))
        mask = (
            tsd[var].notna() &
            ~tsd[var].isin([0, 99])
        )
        ty = tsd.loc[mask].copy()

        # Construir ID_AGE del cuidador reportado en esta variable
        ty["ID_AGE"] = (
            ty["LLAVEHOG"].astype(str)
            + "."
            + ty[var].astype(str).str.replace(r"\.0$", "", regex=True)
        )

        # Merge para obtener la edad del cuidador (all.x = True → left join)
        merged = ty.merge(
            edad_lookup.rename(columns={"EDAD": "EDAD_y"}),
            on="ID_AGE",
            how="left"
        )

        # Renombrar la edad obtenida como <var>_ED
        merged[f"{var}_ED"] = merged["EDAD_y"]

        # Conservar solo LLAVESDE y la columna de edad del cuidador
        frames.append(merged[["LLAVESDE", f"{var}_ED"]].copy())

    return frames


def merge_age_frames(frames):
    """
    Une todos los DataFrames de edades por LLAVESDE (outer join).
    Equivalente a Reduce(function(x,y) merge(x,y,by='LLAVESDE',all=T), l0)
    """
    return reduce(
        lambda left, right: pd.merge(left, right, on="LLAVESDE", how="outer"),
        frames
    )


# ================================================================
# 5. PERSONAS CON DISCAPACIDAD O DEPENDENCIA
#    Variables de cuidador: P4_2_1, P4_2_2, P4_2_3, P4_3
#    Variable de otro hogar: P4_4
# ================================================================
vars_disc = ["P4_2_1", "P4_2_2", "P4_2_3", "P4_3"]

frames_disc = get_caregiver_ages(tsd, vars_disc)
s2 = merge_age_frames(frames_disc)
tsd = tsd.merge(s2, on="LLAVESDE", how="left")

# Construcción del indicador PRC_DISC
# Recibe cuidados si:
#   (a) P4_1==1 (alguien del hogar cuidó) Y al menos un cuidador tiene 15–98 años
#   (b) P4_1==2 (nadie del hogar cuidó) Y el cuidador principal tiene 15–98 años
#   (c) P4_4==1 (recibió cuidados de otro hogar)
tsd["PRC_DISC"] = np.where(
    (tsd["PN_CDISC"] == 1) &
    (
        (
            (tsd["P4_1"] == 1) &
            (
                tsd["P4_2_1_ED"].isin(VALID_AGES) |
                tsd["P4_2_2_ED"].isin(VALID_AGES) |
                tsd["P4_2_3_ED"].isin(VALID_AGES) |
                tsd["P4_3_ED"].isin(VALID_AGES)
            )
        ) |
        (
            (tsd["P4_1"] == 2) &
            tsd["P4_3_ED"].isin(VALID_AGES)
        ) |
        (tsd["P4_4"] == 1)
    ),
    1, 0
)

# ================================================================
# 6. PERSONAS DE 0 A 5 AÑOS
#    Variables de cuidador: P4_17_1, P4_17_2, P4_17_3, P4_18
#    Variable de otro hogar: P4_19
# ================================================================
vars_0a5 = ["P4_17_1", "P4_17_2", "P4_17_3", "P4_18"]

frames_0a5 = get_caregiver_ages(tsd, vars_0a5)
s2 = merge_age_frames(frames_0a5)
tsd = tsd.merge(s2, on="LLAVESDE", how="left")

tsd["PRC_0A5"] = np.where(
    (tsd["PN_C0005"] == 1) &
    (
        (
            (tsd["P4_16"] == 1) &
            (
                tsd["P4_17_1_ED"].isin(VALID_AGES) |
                tsd["P4_17_2_ED"].isin(VALID_AGES) |
                tsd["P4_17_3_ED"].isin(VALID_AGES) |
                tsd["P4_18_ED"].isin(VALID_AGES)
            )
        ) |
        (
            (tsd["P4_16"] == 2) &
            tsd["P4_18_ED"].isin(VALID_AGES)
        ) |
        (tsd["P4_19"] == 1)
    ),
    1, 0
)

# ================================================================
# 7. PERSONAS DE 6 A 17 AÑOS
#    Variables de cuidador: P4_28_1, P4_28_2, P4_28_3, P4_29
#    Variable de otro hogar: P4_30
# ================================================================
vars_6a17 = ["P4_28_1", "P4_28_2", "P4_28_3", "P4_29"]

frames_6a17 = get_caregiver_ages(tsd, vars_6a17)
s2 = merge_age_frames(frames_6a17)
tsd = tsd.merge(s2, on="LLAVESDE", how="left")

tsd["PRC_6A17"] = np.where(
    (tsd["PN_C0617"] == 1) &
    (
        (
            (tsd["P4_27"] == 1) &
            (
                tsd["P4_28_1_ED"].isin(VALID_AGES) |
                tsd["P4_28_2_ED"].isin(VALID_AGES) |
                tsd["P4_28_3_ED"].isin(VALID_AGES) |
                tsd["P4_29_ED"].isin(VALID_AGES)
            )
        ) |
        (
            (tsd["P4_27"] == 2) &
            tsd["P4_29_ED"].isin(VALID_AGES)
        ) |
        (tsd["P4_30"] == 1)
    ),
    1, 0
)

# ================================================================
# 8. PERSONAS DE 60 AÑOS Y MÁS
#    Variables de cuidador: P4_43_1, P4_43_2, P4_43_3, P4_44
#    Variable de otro hogar: P4_45
# ================================================================
vars_60ym = ["P4_43_1", "P4_43_2", "P4_43_3", "P4_44"]

frames_60ym = get_caregiver_ages(tsd, vars_60ym)
s2 = merge_age_frames(frames_60ym)
tsd = tsd.merge(s2, on="LLAVESDE", how="left")

tsd["PRC_60YM"] = np.where(
    (tsd["PN_C60MA"] == 1) &
    (
        (
            (tsd["P4_42"] == 1) &
            (
                tsd["P4_43_1_ED"].isin(VALID_AGES) |
                tsd["P4_43_2_ED"].isin(VALID_AGES) |
                tsd["P4_43_3_ED"].isin(VALID_AGES) |
                tsd["P4_44_ED"].isin(VALID_AGES)
            )
        ) |
        (
            (tsd["P4_42"] == 2) &
            tsd["P4_44_ED"].isin(VALID_AGES)
        ) |
        (tsd["P4_45"] == 1)
    ),
    1, 0
)

# ================================================================
# 9. PERSONAS ENFERMAS TEMPORALES
#    Variables de cuidador: P4_59_1, P4_59_2, P4_59_3, P4_60
#    Variable de otro hogar: P4_61
# ================================================================
vars_tem = ["P4_59_1", "P4_59_2", "P4_59_3", "P4_60"]

frames_tem = get_caregiver_ages(tsd, vars_tem)
s2 = merge_age_frames(frames_tem)
tsd = tsd.merge(s2, on="LLAVESDE", how="left")

tsd["PRC_TEM"] = np.where(
    (tsd["PN_CETEM"] == 1) &
    (
        (
            (tsd["P4_58"] == 1) &
            (
                tsd["P4_59_1_ED"].isin(VALID_AGES) |
                tsd["P4_59_2_ED"].isin(VALID_AGES) |
                tsd["P4_59_3_ED"].isin(VALID_AGES) |
                tsd["P4_60_ED"].isin(VALID_AGES)
            )
        ) |
        (
            (tsd["P4_58"] == 2) &
            tsd["P4_60_ED"].isin(VALID_AGES)
        ) |
        (tsd["P4_61"] == 1)
    ),
    1, 0
)

# ================================================================
# 10. POBLACIÓN SUSCEPTIBLE QUE RECIBE CUIDADOS (indicador final)
# ================================================================

# EXCLUYE enfermos temporales
tsd["PRC_EX_CETEM"] = np.where(
    (tsd["PRC_DISC"] == 1) |
    (tsd["PRC_0A5"]  == 1) |
    (tsd["PRC_6A17"] == 1) |
    (tsd["PRC_60YM"] == 1),
    1, 0
)

# INCLUYE enfermos temporales
tsd["PRC_IN_CETEM"] = np.where(
    (tsd["PRC_DISC"] == 1) |
    (tsd["PRC_0A5"]  == 1) |
    (tsd["PRC_6A17"] == 1) |
    (tsd["PRC_60YM"] == 1) |
    (tsd["PRC_TEM"]  == 1),
    1, 0
)

# ================================================================
# 11. Limpieza: eliminar columnas auxiliares
#     Equivalente a rm(ty,l0,s2,xd) y tsd=tsd[,!names...]
# ================================================================
cols_to_drop = ["N_REN", "ID_AGE"]
tsd = tsd.drop(columns=[c for c in cols_to_drop if c in tsd.columns])

# ================================================================
# 12. Verificación rápida de resultados
# ================================================================
print("=" * 60)
print("POBLACIÓN SUSCEPTIBLE QUE RECIBE CUIDADOS — ENASIC 2022")
print("=" * 60)

grupos = {
    "Discapacidad / dependencia": "PRC_DISC",
    "Niñas y niños 0–5 años":     "PRC_0A5",
    "Niñas, niños y adol. 6–17":  "PRC_6A17",
    "Adultas/os mayores 60+":      "PRC_60YM",
    "Enfermas/os temporales":      "PRC_TEM",
}

for label, col in grupos.items():
    n = tsd[col].sum()
    print(f"  {label:<35}: {n:>7,} registros con PRC=1")

print("-" * 60)
print(f"  {'PRC_EX_CETEM (sin temporales)':<35}: "
      f"{tsd['PRC_EX_CETEM'].sum():>7,}")
print(f"  {'PRC_IN_CETEM (con temporales)':<35}: "
      f"{tsd['PRC_IN_CETEM'].sum():>7,}")
print("=" * 60)
print(f"\nDimensiones finales del DataFrame: {tsd.shape}")

/tmp/ipykernel_36795/1050282541.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["N_REN"] = (
/tmp/ipykernel_36795/1050282541.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["ID_AGE"] = (
/tmp/ipykernel_36795/1050282541.py:141: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["PRC_D

POBLACIÓN SUSCEPTIBLE QUE RECIBE CUIDADOS — ENASIC 2022
  Discapacidad / dependencia         :     621 registros con PRC=1
  Niñas y niños 0–5 años             :   1,795 registros con PRC=1
  Niñas, niños y adol. 6–17          :   3,536 registros con PRC=1
  Adultas/os mayores 60+             :     580 registros con PRC=1
  Enfermas/os temporales             :     142 registros con PRC=1
------------------------------------------------------------
  PRC_EX_CETEM (sin temporales)      :   6,532
  PRC_IN_CETEM (con temporales)      :   6,620

Dimensiones finales del DataFrame: (21776, 208)


/tmp/ipykernel_36795/1050282541.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["PRC_TEM"] = np.where(
/tmp/ipykernel_36795/1050282541.py:295: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["PRC_EX_CETEM"] = np.where(
/tmp/ipykernel_36795/1050282541.py:304: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = 

In [11]:



######################################################################
# 1. Cargar base de datos
######################################################################




######################################################################
# 2. Construcción de CUI_G
######################################################################

# Variables P4_69_1 a P4_69_6
cui_cols = [f"P4_69_{i}" for i in range(1, 7)]

# Convertir a numérico por seguridad
tsd[cui_cols] = tsd[cui_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# R:
# z = paste0("P4_69_", 1:6)
# aux1 = P4_69_1 %in% c(1,9) |
#        P4_69_2 %in% c(1,9) | ...
# CUI_G = 1 si alguna variable toma 1 o 9

tsd["CUI_G"] = (
    tsd[cui_cols]
    .isin([1, 9])
    .any(axis=1)
    .astype(int)
)


######################################################################
# 3. Población de 15 años y más
######################################################################

tsd["EDAD"] = pd.to_numeric(
    tsd["EDAD"],
    errors="coerce"
)

tsd["FAC_HOG"] = pd.to_numeric(
    tsd["FAC_HOG"],
    errors="coerce"
)

# R:
# a1 = ifelse(EDAD %in% 15:98, FAC_HOG, 0)

tsd["a1"] = tsd["FAC_HOG"].where(
    tsd["EDAD"].between(15, 98, inclusive="both"),
    0
)


######################################################################
# 4. Condición de brindar o no apoyo
######################################################################

# R:
# a2 = ifelse(
#     !a1 %in% 0 & CUI_G %in% 1,
#     FAC_HOG,
#     0
# )

tsd["a2"] = tsd["FAC_HOG"].where(
    tsd["a1"].ne(0) & tsd["CUI_G"].eq(1),
    0
)

# R:
# a3 = ifelse(
#     !a1 %in% 0 & CUI_G %in% 0,
#     FAC_HOG,
#     0
# )

tsd["a3"] = tsd["FAC_HOG"].where(
    tsd["a1"].ne(0) & tsd["CUI_G"].eq(0),
    0
)


######################################################################
# 5. Validar variables del diseño
######################################################################

design_cols = [
    "UPM_DIS",
    "EST_DIS"
]

missing_design = tsd[design_cols].isna().any()

if missing_design.any():
    raise ValueError(
        "Existen valores NA en UPM_DIS o EST_DIS."
    )


######################################################################
# 6. Función para calcular la varianza con
#    conglomerados últimos + estratificación
######################################################################

def survey_variance(
    df,
    value_cols,
    psu_col="UPM_DIS",
    strata_col="EST_DIS"
):
    """
    Calcula la varianza tipo 'ultimate cluster'
    utilizada por survey::svytotal / svyratio
    para un diseño estratificado por PSU.

    La lógica es:

        1. Sumar las observaciones dentro de cada PSU.
        2. Obtener la media de PSU dentro de cada estrato.
        3. Centrar cada PSU respecto a su estrato.
        4. Aplicar n_h / (n_h - 1).
        5. Sumar entre estratos.

    No incorpora FPC porque el código R original
    tampoco especifica fpc.
    """

    # Total por PSU dentro de cada estrato
    cluster_totals = (
        df.groupby(
            [strata_col, psu_col],
            sort=False,
            dropna=False
        )[value_cols]
        .sum(min_count=1)
    )

    # Media de los totales PSU dentro de cada estrato
    strata_means = (
        cluster_totals
        .groupby(level=0)[value_cols]
        .transform("mean")
    )

    # Desviaciones respecto a la media del estrato
    centered = cluster_totals - strata_means

    # Suma de cuadrados dentro de cada estrato
    ss = (
        centered.pow(2)
        .groupby(level=0)
        .sum()
    )

    # Número de PSU por estrato
    n_psu = (
        cluster_totals
        .groupby(level=0)
        .size()
    )

    # Igual que survey::survey.lonely.psu = "fail"
    lonely = n_psu[n_psu <= 1]

    if not lonely.empty:
        raise ValueError(
            "Hay estratos con una sola UPM_DIS. "
            "El comportamiento por defecto de R survey "
            "es detenerse ante estos estratos."
        )

    # Corrección n_h / (n_h - 1)
    correction = n_psu / (n_psu - 1)

    variance = (
        ss
        .mul(correction, axis=0)
        .sum(axis=0)
    )

    return variance


######################################################################
# 7. Estimaciones totales
######################################################################

variables = [
    "a1",
    "a2",
    "a3"
]

# Equivalente a:
#
# e1 = svytotal(~a1 + a2 + a3, diseno)
#
# Como weight = 1:
# total = sum(variable)

estimaciones = tsd[variables].sum()

# Varianzas de los totales
var_totales = survey_variance(
    tsd,
    variables,
    psu_col="UPM_DIS",
    strata_col="EST_DIS"
)

# Error estándar
se_totales = np.sqrt(var_totales)


######################################################################
# 8. Coeficiente de variación de los totales
######################################################################

cv_totales = (
    se_totales / estimaciones
)


######################################################################
# 9. Estimaciones relativas
######################################################################

# Equivalente a:
#
# e11 = svyratio(
#     ~a1 + a2 + a3,
#     denominator = ~a1,
#     diseno
# )

denominador = estimaciones["a1"]

ratios = estimaciones / denominador


######################################################################
# 10. Varianza de los cocientes
######################################################################

# Para svyratio, survey construye:
#
# r = (numerador - R * denominador) /
#     sum(denominador / prob)
#
# Como prob = 1:
#
# r = (numerador - R * denominador) / sum(denominador)

residuos_ratio = pd.DataFrame(
    index=tsd.index
)

for variable in variables:

    residuos_ratio[variable] = (
        tsd[variable]
        - ratios[variable] * tsd["a1"]
    ) / denominador


######################################################################
# 11. Varianza de los ratios
######################################################################

var_ratios = survey_variance(
    pd.concat(
        [
            tsd[["UPM_DIS", "EST_DIS"]],
            residuos_ratio
        ],
        axis=1
    ),
    variables,
    psu_col="UPM_DIS",
    strata_col="EST_DIS"
)

se_ratios = np.sqrt(var_ratios)

cv_ratios = (
    se_ratios / ratios
)


######################################################################
# 12. Intervalos de confianza al 90%
######################################################################

# confint() en survey usa df = Inf por defecto.
#
# Para 90%:
#
# alpha = 0.10
# z_(1-alpha/2) = z_0.95

z90 = norm.ppf(0.95)

ici_totales = (
    estimaciones - z90 * se_totales
)

ics_totales = (
    estimaciones + z90 * se_totales
)

ici_ratios = (
    ratios - z90 * se_ratios
)

ics_ratios = (
    ratios + z90 * se_ratios
)


######################################################################
# 13. Construcción de la tabla de resultados
######################################################################

com = [
    "Estados Unidos Mexicanos",

    "Sí brinda apoyo o cuidados a personas "
    "del hogar u otros hogares",

    "No brinda apoyo o cuidados a personas "
    "del hogar u otros hogares"
]


resultados = pd.DataFrame(
    {
        "EST": estimaciones.values,
        "REL": ratios.values * 100,
        "CV": cv_totales.values * 100,
        "CVREL": cv_ratios.values * 100,
        "SE": se_totales.values,
        "SEREL": se_ratios.values * 100,
        "ICI": ici_totales.values,
        "ICIREL": ici_ratios.values * 100,
        "ICS": ics_totales.values,
        "ICSREL": ics_ratios.values * 100
    }
)


######################################################################
# 14. Agregar etiquetas
######################################################################

m1 = pd.concat(
    [
        pd.Series(com, name="Concepto"),
        resultados
    ],
    axis=1
)


######################################################################
# 15. Crear formato de salida equivalente a Tabulado
######################################################################

# El código R inserta columnas vacías entre grupos de indicadores.

tabulado = pd.DataFrame(
    {
        "Concepto": com,
        "": ["", "", ""],

        "EST": resultados["EST"],
        "REL": resultados["REL"],

        "  ": ["", "", ""],

        "SE": resultados["SE"],
        "SEREL": resultados["SEREL"],

        "   ": ["", "", ""],

        "CV": resultados["CV"],
        "CVREL": resultados["CVREL"],

        "    ": ["", "", ""],

        "ICI": resultados["ICI"],
        "ICIREL": resultados["ICIREL"],

        "     ": ["", "", ""],

        "ICS": resultados["ICS"],
        "ICSREL": resultados["ICSREL"]
    }
)


######################################################################
# 16. Exportar a Excel
######################################################################

archivo_salida = "Tabulado_de_cuidados.xlsx"

with pd.ExcelWriter(
    archivo_salida,
    engine="openpyxl"
) as writer:

    tabulado.to_excel(
        writer,
        sheet_name="Tabulado",
        index=False
    )

    # Ajustar ancho de columnas
    worksheet = writer.book["Tabulado"]

    for column_cells in worksheet.columns:

        max_length = 0
        column_letter = column_cells[0].column_letter

        for cell in column_cells:

            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        worksheet.column_dimensions[
            column_letter
        ].width = min(max_length + 2, 45)


######################################################################
# 17. Mostrar resultado
######################################################################

print(tabulado)
print(f"\nArchivo generado: {archivo_salida}")



/tmp/ipykernel_36795/3246163604.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["CUI_G"] = (
/tmp/ipykernel_36795/3246163604.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["a1"] = tsd["FAC_HOG"].where(
/tmp/ipykernel_36795/3246163604.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

                                            Concepto         EST         REL  \
0                           Estados Unidos Mexicanos    98924781  100.000000   
1  Sí brinda apoyo o cuidados a personas del hoga...    31652134   31.996163   
2  No brinda apoyo o cuidados a personas del hoga...    67272647   68.003837   

                SE     SEREL            CV     CVREL                ICI  \
0     1.314502e+06  0.000000      1.328789  0.000000       9.676262e+07   
1     6.236804e+05  0.513492      1.970421  1.604856       3.062627e+07   
2     1.074607e+06  0.513492      1.597391  0.755093       6.550508e+07   

       ICIREL                 ICS      ICSREL  
0  100.000000        1.010869e+08  100.000000  
1   31.151543        3.267800e+07   32.840782  
2   67.159218        6.904022e+07   68.848457  

Archivo generado: Tabulado_de_cuidados.xlsx


In [13]:



######################################################################
# 1. Cargar base de datos
######################################################################




######################################################################
# 2. Construir identificador de cada persona
######################################################################

# LLAVESDE contiene el número de renglón al final.
# Se extraen los últimos 2 caracteres.

tsd["N_REN"] = pd.to_numeric(
    tsd["LLAVESDE"].astype(str).str[-2:],
    errors="coerce"
)

# ID único = hogar + número de renglón
tsd["ID_CP"] = (
    tsd["LLAVEHOG"].astype(str)
    + "."
    + tsd["N_REN"].astype("Int64").astype(str)
)


######################################################################
# 3. Definir las poblaciones cuidadoras
######################################################################

poblaciones = {
    "CUI_DISC": [
        "P4_2_1",
        "P4_2_2",
        "P4_2_3",
        "P4_3"
    ],

    "CUI_0A5": [
        "P4_17_1",
        "P4_17_2",
        "P4_17_3",
        "P4_18"
    ],

    "CUI_6A17": [
        "P4_28_1",
        "P4_28_2",
        "P4_28_3",
        "P4_29"
    ],

    "CUI_60YM": [
        "P4_43_1",
        "P4_43_2",
        "P4_43_3",
        "P4_44"
    ],

    "CUI_TEM": [
        "P4_59_1",
        "P4_59_2",
        "P4_59_3",
        "P4_60"
    ]
}


######################################################################
# 4. Construir cada población cuidadora
######################################################################

for nombre, variables in poblaciones.items():

    # Conjunto de IDs de personas identificadas como cuidadoras
    ids_cuidadores = set()

    for variable in variables:

        # Valores válidos:
        # - No NA
        # - Diferentes de 0
        # - Diferentes de 99

        mask = (
            tsd[variable].notna()
            & ~tsd[variable].isin([0, 99])
        )

        # ID del cuidador:
        # LLAVEHOG + número de renglón indicado por la variable

        ids = (
            tsd.loc[mask, "LLAVEHOG"].astype(str)
            + "."
            + tsd.loc[mask, variable].astype(str)
        )

        ids_cuidadores.update(ids)

    # Marcar a las personas que aparecen como cuidadoras
    tsd[nombre] = tsd["ID_CP"].isin(ids_cuidadores).astype(int)


######################################################################
# 5. Personas cuidadoras de otro hogar
######################################################################

tsd["CUI_OHOG"] = tsd["P4_68"].eq(1).astype(int)


######################################################################
# 6. Población cuidadora general
######################################################################

columnas_cuidador = [
    "CUI_DISC",
    "CUI_0A5",
    "CUI_6A17",
    "CUI_60YM",
    "CUI_TEM",
    "CUI_OHOG"
]

tsd["CUI_G"] = (
    tsd[columnas_cuidador]
    .eq(1)
    .any(axis=1)
    .astype(int)
)


######################################################################
# 7. Eliminar variables auxiliares
######################################################################

tsd.drop(
    columns=["N_REN", "ID_CP"],
    inplace=True
)


######################################################################
# Resultado:
# tsd contiene las variables:
#
# CUI_DISC
# CUI_0A5
# CUI_6A17
# CUI_60YM
# CUI_TEM
# CUI_OHOG
# CUI_G
######################################################################



/tmp/ipykernel_36795/2534646839.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["N_REN"] = pd.to_numeric(
/tmp/ipykernel_36795/2534646839.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tsd["ID_CP"] = (
/tmp/ipykernel_36795/2534646839.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [14]:
tsd

,LLAVESDE,LLAVEVIV,LLAVEHOG,PAREN,PAREN_C,SEXO,EDAD,P3_6,P3_6A,P3_7_1,...,CUI_G,a1,a2,a3,CUI_DISC,CUI_0A5,CUI_6A17,CUI_60YM,CUI_TEM,CUI_OHOG
0,101101,101,1011,1,NaN,1,56,1,NaN,1,...,0,1593,0,1593,0,0,0,0,0,0
1,101102,101,1011,2,NaN,2,54,1,NaN,1,...,0,1593,0,1593,0,0,0,0,0,0
2,101103,101,1011,3,NaN,2,22,1,NaN,1,...,0,1593,0,1593,0,0,0,0,0,0
3,101104,101,1011,3,NaN,1,21,1,NaN,1,...,0,1593,0,1593,0,0,0,0,0,0
4,102101,102,1021,1,NaN,2,72,1,NaN,1,...,0,1593,0,1593,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21771,89719101,89719,897191,1,NaN,2,53,6,NaN,2,...,0,2135,0,2135,0,0,0,0,0,0
21772,89719102,89719,897191,3,NaN,2,33,10,2.0,1,...,0,2135,2135,0,0,0,0,0,0,0
21773,89719103,89719,897191,4,NaN,2,13,10,2.0,1,...,0,0,0,0,0,0,0,0,0,0
21774,89720101,89720,897201,1,NaN,1,54,10,2.0,2,...,0,2135,0,2135,0,0,0,0,0,0
